# An Affordable Recipe for Training Diffusion Transformers

### Dog-breed text-to-image diffusion, from pretraining to fine-tuning, on 4 consumer GPUs

*A worked example covering latent-space diffusion, flow matching, representation alignment
(REPA), and the practical realities of fine-tuning on a small dataset — including a real case
where FID rises during fine-tuning, and why that's a reference-distribution artifact worth
understanding rather than a straightforward quality regression.*

> **The headline result.** Training diffusion transformers is usually presented as something
> that requires industrial-scale compute. The recipe below is the result of two months of
> empirically comparing image tokenizers, latent-space parameterizations, and training
> hyperparameters — converging on a configuration that pretrains a working text-to-image
> Diffusion Transformer (DiT) to **sub-10 FID in 2.5 hours**, then fine-tunes it in **20 more
> minutes**, on **4x RTX 3090 (24GB)** — hardware you can rent by the hour. At quoted Aug-2026
> rates (Vast.ai \$0.25/GPU/hr, RunPod \$0.46/GPU/hr), that's roughly **\$3–\$5 of total
> rented compute** for the whole pipeline (Experiments 1 and 2 below). This notebook teaches
> both *why* the pipeline works and *how* that specific recipe was found, so you can run it —
> and adapt it — yourself.

**What this notebook covers.** We built a two-stage text-to-image pipeline: a frozen visual
tokenizer (VAE) that compresses images to a small latent grid, and a Diffusion Transformer
(DiT) trained with flow matching to generate new latents conditioned on a text caption. We
pretrained the DiT from scratch on ~26,000 recaptioned dog images, then fine-tuned (SFT) the
result on a smaller, curated synthetic dataset of 20 dog breeds. Every concept below is
explained from first principles and tied to the actual code and configs used — nothing here
is a toy reimplementation, this is a walkthrough of a real research pipeline and its real
results, including the parts that didn't go as expected.

**Prerequisites.** Comfort with PyTorch and `nn.Module`; a general sense of what a neural
network training loop looks like. No prior diffusion-model or transformer-internals
knowledge assumed — those are introduced from scratch in Sections 2–3.

**Compute.** This walkthrough assumes access similar to what was actually used for these
experiments: a multi-GPU machine (the real runs used 4x RTX 3090, 24GB each — the same
hardware behind the headline result above). The commands shown are the *actual* commands
used, not a scaled-down toy version — adjust `--nproc_per_node` / `CUDA_VISIBLE_DEVICES` and
batch size to match whatever GPU count you have; everything else in the pipeline works
unchanged at smaller scale, just slower. If renting rather than using an existing machine,
4x RTX 3090 for a few hours costs on the order of single-digit dollars at current spot rates
(see the callout above) — training a text-to-image diffusion transformer from scratch is not
an industrial-scale undertaking with this recipe.

**No pretrained weights are provided with this notebook, deliberately.** This is meant to be
*run*, not inspected: the raw datasets are provided directly (Section 8.1) so your compute
budget goes toward the actual training, not data collection — Section 9 has you pretrain
your own base model from scratch, and Section 10 has you fine-tune your own SFT checkpoint
and plot *your own* evaluation curve, not a pre-supplied one. Everything downstream of the
raw data — the latent cache, both checkpoints, both sets of results — is entirely your own.


## 0. Setup & configuration

Clone the repo and install dependencies:

```bash
git clone https://github.com/Reyhaneesmailizadeh/diffusion-bench.git
cd diffusion-bench
curl -LsSf https://astral.sh/uv/install.sh | sh   # install uv, if you don't have it
uv sync                                            # reproduces the exact Python environment
```

Every path below is a placeholder pointing at *your own* working directory — nothing here
resolves to any pre-existing data or checkpoints. Fill these in once and every later cell in
this notebook reuses them, and every shell command in this notebook uses the equivalent
`$DOGS_ROOT`-style variable so you can copy-paste them directly into a terminal too.


In [ ]:
from pathlib import Path

REPO_ROOT = Path("/path/to/your/diffusion-bench")     # <-- where you cloned the repo
DOGS_ROOT = Path("/path/to/your/data/dogs")            # <-- where you'll keep all dog data + caches
CKPT_ROOT = Path("/path/to/your/ckpts")                # <-- where trained checkpoints will be written

# Sub-paths used throughout the notebook -- populated as you work through Section 8.
RECAPTIONED_WDS_DIR  = DOGS_ROOT / "dogs_recaptioned_wds"          # downloaded in Section 8.1
PRETRAIN_LATENTS_DIR = DOGS_ROOT / "dogs_recaptioned_latents_wds"  # after scripts/precompute_latents.py

SFT_WDS_DIR      = DOGS_ROOT / "dogs_sft_wds"                      # downloaded in Section 8.1
SFT_LATENTS_DIR  = DOGS_ROOT / "dogs_sft_latents_wds"

for p in [REPO_ROOT, DOGS_ROOT, CKPT_ROOT]:
    print(f"{p}  {'(exists)' if p.exists() else '<-- create this / point it somewhere real before continuing'}")


## 1. Pipeline overview

```
raw images + captions                 (jpg + txt pairs, WebDataset shards)
        │
        │  scripts/precompute_latents.py   (Section 8)
        ▼
precomputed cache: VAE latents + text embeddings + DINOv2 features, per sample
        │
        │  Stage-2 training (flow matching + REPA loss)          (Sections 6-7)
        ▼
Experiment 1: pretrain LightningDiT from scratch, 200 epochs      (Section 9)
        │
        │  warm-start from the pretrained checkpoint, fine-tune on
        │  a smaller domain-specific dataset
        ▼
Experiment 2: SFT on dogs-synthetic-2k, 100 epochs                (Section 10)
        │
        ▼
inference: generate images from text captions, evaluate FID/IS    (Section 11)
```

**Why two stages (VAE + DiT) instead of one model working directly on pixels?** A 256×256
RGB image is 196,608 numbers. Running a transformer directly over that many positions is
prohibitively expensive. A VAE (Stage 1, described in Section 3) is trained once to compress
an image into a much smaller *latent* grid — here, 16×16×32 — that still contains enough
information to reconstruct a visually near-identical image. The DiT (Stage 2) then only has
to learn to generate 16×16=256 latent tokens instead of 196,608 pixels, which is what makes
training a transformer-based image generator tractable at all. This is the same idea behind
Stable Diffusion and most modern latent diffusion models.

**Why two *training* stages (pretrain + SFT)?** The pretrained model learns general
"what does a dog look like, and how do I follow a caption" from a large, broad dataset. The
SFT stage then specializes that general ability onto a smaller, cleaner, more specifically
curated dataset (20 hand-picked breeds, synthetic images). This is the same
pretrain-then-fine-tune pattern used throughout modern deep learning (e.g. LLM pretraining
+ instruction tuning) — and, as Section 10 shows with real data, it comes with a real gotcha:
your evaluation metric can move in a direction that looks bad but isn't, if the fine-tuning
set has a different image distribution than whatever your metric is comparing against.


## 2. Background: what is a diffusion / flow-matching model?

**The core idea.** We want a model that can turn random noise into a realistic image (or, in
our case, a realistic image *matching a caption*). The trick used by diffusion and
flow-matching models: instead of trying to jump straight from noise to a finished image, we
train a network to make many small, easy steps that gradually turn noise into data.

**Flow matching, concretely.** Let $x_1$ be a real data sample (a VAE latent, in our case)
and $x_0$ be pure Gaussian noise of the same shape. Define a straight-line path between them,
indexed by a "time" $t \in [0, 1]$:

$$x_t = (1-t)\, x_1 + t\, x_0$$

At $t=0$ this is the clean data; at $t=1$ it's pure noise. The *velocity* of a particle
moving along this path is just the derivative:

$$v_t = \frac{d x_t}{dt} = x_0 - x_1$$

We train a neural network $v_\theta(x_t, t, \text{caption})$ to predict this velocity at
random points along random paths. This is *exactly* what happens in this codebase's
`Transport.sample()` and `Transport.compute_loss()`
(`src/stage2/transport/transport.py`):

```python
def sample(self, x1):
    x0 = th.randn_like(x1)
    t = self.time_sampler(x1.shape[0]).to(x1)          # NOT uniform -- see below
    ...
    return t, x0, x1

# elsewhere, inside training_losses():
xt = (1 - t) * x1 + t * x0
vt = (xt - x1) / t.clamp_min(self.t_eps)                # the velocity target
...
loss = (model_output - vt) ** 2                          # simple MSE against the prediction
```

Once trained, generating an image means starting from pure noise ($t=1$) and numerically
integrating the learned velocity field backward to $t=0$ in a handful of discrete steps (an
ODE solver) — this is what `sampler.num_steps: 50` in the configs controls.

**Why isn't $t$ sampled uniformly?** Not all noise levels are equally informative to train
on — near-pure-noise steps ($t$ close to 1) carry very little signal about the target image,
while some middle range is empirically the most useful to spend training compute on. This
codebase samples $t$ from a **logit-normal** distribution
(`time_dist_type: 'logit-normal_0_1'` in the configs) rather than uniformly, concentrating
training on the more informative middle range of the noise schedule.

**Classifier-free guidance (CFG).** At inference time, we want generations to follow the
caption *strongly*. The trick: train the model to also handle an *empty* caption some
fraction of the time (`cfg_dropout_prob: 0.1` — 10% of training steps drop the caption
entirely), then at generation time, extrapolate *away* from the unconditional prediction and
*towards* the conditional one:

$$v_{\text{guided}} = v_{\text{uncond}} + s \cdot (v_{\text{cond}} - v_{\text{uncond}})$$

with guidance scale $s > 1$ (`guidance.cfg.scale: 6.0` in these configs) pushing generations
to adhere to the prompt more strongly than a plain conditional prediction would.


## 3. Model component: the visual tokenizer (Stage 1 VAE)

The VAE (`src/stage1/vae.py`) is a **frozen**, pretrained autoencoder — it is never trained
as part of this project, only used to encode images into latents and decode latents back
into images. It wraps a HuggingFace `diffusers.AutoencoderKL` checkpoint. The specific
variant used throughout, `e2e-invae`, resolves to the pretrained weights
`REPA-E/e2e-invae-hf` (`VAE_CONFIGS` in `vae.py`) — a VAE whose latent space was itself
trained end-to-end with a REPA-style alignment objective against a self-supervised vision
encoder, which is a separate, unrelated-to-us detail of how *that* checkpoint was produced;
we just consume it as a fixed component.

Key numbers: 32 latent channels, 16× spatial downsampling — so a 256×256×3 image becomes a
16×16×32 latent (which is exactly `misc.latent_size: [32, 16, 16]` in the stage-2 configs).

**Why this specific tokenizer?** The choice of visual tokenizer is not incidental to the
headline result at the top of this notebook — it's one of the main knobs that was varied
during the two months of comparisons that produced this recipe. A tokenizer's latent
channel count and spatial downsampling ratio directly set how many tokens the DiT has to
learn to generate (here, 16×16=256), which is a first-order driver of how much compute a
given FID target costs to reach. `e2e-invae` was the compressor that came out of that
comparison as the best fit for training a small DiT quickly; the tokenizer is swappable
(see `VAE_CONFIGS` in `vae.py`) if you want to reproduce that comparison yourself.

Exact source, `src/stage1/vae.py`:

```python
@torch.no_grad()
def encode(self, x: torch.Tensor) -> torch.Tensor:
    x = self._preprocess(x)
    posterior = self._vae_encode(x)
    z = posterior.sample() if self.sample_mode == "sample" else posterior.mode()
    z = (z - self.shift_factor) * self.scaling_factor
    return z

def decode(self, z: torch.Tensor) -> torch.Tensor:
    z = z / self.scaling_factor + self.shift_factor
    x = self._vae_decode(z)
    return ((x + 1.0) / 2.0).clamp(0, 1)  # [-1, 1] -> [0, 1]
```

Let's actually load it and see it work.


In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
from torchvision.utils import make_grid
from torchvision.transforms.functional import to_pil_image

from utils.model_utils import instantiate_from_config
from omegaconf import OmegaConf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

stage1_cfg = OmegaConf.create({
    "target": "stage1.VAE",
    "params": {"vae_type": "e2e-invae", "resolution": 256},
})
rae = instantiate_from_config(stage1_cfg).to(device).eval()
print(f"Stage-1 VAE loaded. Latent channels: {rae.latent_dim}, downsample factor: {rae.patch_size}")


In [ ]:
# Encode one of your own images and decode it straight back, to see how much visual
# information survives the 256x256x3 -> 16x16x32 compression. Pulls a single sample
# straight out of a WebDataset shard -- run Section 8.1 first so $RECAPTIONED_WDS_DIR
# actually has shards in it, or point shard_path at any other .tar you already have.
import io
import tarfile
from PIL import Image
from torchvision import transforms

shard_path = next(RECAPTIONED_WDS_DIR.glob("*.tar"))  # <-- or set this to a specific shard
with tarfile.open(shard_path) as tf:
    jpg_member = next(m for m in tf.getmembers() if m.name.endswith(".jpg"))
    img = Image.open(io.BytesIO(tf.extractfile(jpg_member).read())).convert("RGB")
print(f"using: {shard_path.name} :: {jpg_member.name}")

transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # [0, 1]
])
x = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    z = rae.encode(x)
    x_rec = rae.decode(z)

print(f"image shape: {tuple(x.shape)}  ->  latent shape: {tuple(z.shape)}  ->  reconstructed: {tuple(x_rec.shape)}")
print(f"compression ratio: {x.numel() / z.numel():.1f}x fewer numbers in latent space")

grid = make_grid(torch.cat([x, x_rec], dim=0), nrow=2)
to_pil_image(grid.cpu())


## 4. Model component: text conditioning (Qwen3-0.6B)

Captions are encoded with **Qwen3-0.6B** (`conditioning.text_encoder.model_name` in the
configs), a small language model, used purely as a text encoder here (not for generation).
Two choices worth calling out because they differ from common practice (e.g. plain CLIP text
encoders):

- **Per-token, not pooled.** The model keeps *all* token hidden states
  (`max_length: 128` tokens), not a single pooled embedding. This gives the DiT much more
  granular conditioning information to attend to — which specific word corresponds to which
  part of the image can, in principle, be resolved token-by-token rather than being
  collapsed into one vector up front.
- **A causal LLM as a text encoder** rather than a model trained specifically for
  cross-modal alignment (like CLIP) — the hidden states it produces are optimized for
  language modeling, not for being "reachable" from images. Whether this matters much in
  practice is exactly the kind of question this pipeline's REPA and attention-alignment
  experiments try to probe.


In [ ]:
from stage2.utils import setup_text_encoder
from configs.stage2 import Stage2Config, ConditioningConfig, TextEncoderConfig

text_cfg = Stage2Config()
text_cfg.conditioning = ConditioningConfig(
    type="text",
    text_encoder=TextEncoderConfig(model_name="Qwen/Qwen3-0.6B", max_length=128),
)
text_encoder = setup_text_encoder(text_cfg, rank=0, device=device)

caption = "Afghan Hound, adult, slender build, long silky golden coat, standing in a grassy field."
with torch.no_grad():
    enc_out = text_encoder([caption])

print(f"tokens shape: {tuple(enc_out['tokens'].shape)}   (batch, seq_len=128, hidden_dim)")
print(f"attention_mask shape: {tuple(enc_out['attention_mask'].shape)}")
print(f"real (non-padding) tokens in this caption: {int(enc_out['attention_mask'].sum())}")


## 5. Model component: REPA (representation alignment)

**The idea.** DINOv2 (Oquab et al., 2023) is a self-supervised vision transformer whose
learned features are known to be strongly, cleanly semantic — similar objects/parts end up
with similar features, without ever being trained for a specific downstream task. REPA
(Yu et al., "Representation Alignment for Generation," 2024) observed that diffusion
transformers train noticeably faster and generate better images if you add an auxiliary loss
pulling one of the DiT's *own intermediate hidden states* toward DINOv2's features of the
*clean* target image — effectively giving the DiT's early/middle layers a semantic "head
start" it would otherwise have to discover purely from the denoising objective.

**In this codebase**, `Transport.training_losses` (`src/stage2/transport/transport.py`) calls
the DiT with `return_intermediate=True`, which makes it also return `zt_pred` — its own
hidden state at layer `repa_layer_depth`, projected to DINOv2's feature dimension via a small
linear head (`self.repa_projector` in `LightningDiT`). The loss itself is the exact source:

```python
loss_repa = th.tensor(0.0, device=x1.device)
if enable_repa and zt_pred is not None:
    loss_repa = repa_coeff * F.mse_loss(zt_pred, z_clean)
terms['loss_repa'] = loss_repa
```

`z_clean` is DINOv2's real patch-token features of the *clean* target image, computed once
and cached during Section 8's precompute step, not recomputed live during training.

Both pretraining and SFT configs here use `repa_layer_depth: 4` (align at the 4th of 12
DiT layers) and `repa_coeff: 0.5` (weight of this auxiliary loss relative to the main
denoising loss), with `target_encoder: dinov2-vit-bnormworeg`.


## 6. Model component: LightningDiT — the generative backbone

This is the actual transformer being trained. Standard DiT ingredients: patchify the latent
into a sequence of tokens (here `patch_size: 1`, so each of the 16×16 latent's spatial
positions is already one token — 256 image tokens total), run them through transformer
blocks, unpatchify the output back into a latent-shaped velocity prediction.

**The one big architectural choice worth dwelling on: how is the model conditioned on time
and text?** The original DiT paper (Peebles & Xie, 2022) uses *AdaLN-Zero*: a small MLP
predicts per-block scale/shift/gate values from the conditioning vector, and every
LayerNorm in the network is modulated by those values. **This codebase does not do that.**
Instead it uses **in-context conditioning**: the time embedding and the (per-token!) text
embedding are turned into extra *tokens* and literally concatenated onto the sequence of
image tokens:

Exact source, `src/stage2/models/lightningDiT.py` (`enable_reg` and `use_cfg_conds` are both
`False` in every config used in this notebook, so in practice this reduces to image tokens +
time tokens + text tokens, concatenated in that order):

```python
def _build_sequence(self, x, t, condition_kwargs):
    seq = []
    if self.enable_reg:
        seq.append(self.cls_in_norm(self.cls_in_proj(condition_kwargs["cls_t"])).unsqueeze(1))
    seq.append(self.x_embedder(x))
    seq.append(self.t_embedder(t))
    if self.use_cfg_conds:
        seq.append(self.cfg_w_embedder(condition_kwargs["omega"]))
    seq.append(self.ctx_embedder(condition_kwargs["context"]))
    seq = torch.cat(seq, dim=1)
    return seq
```

The whole concatenated sequence then goes through **plain, unconditional** RMSNorm and
**joint self-attention** — every block's attention is one shared softmax over image, time,
*and* text tokens together, with no separate cross-attention module and no per-block
modulation. This has a nice side effect: because time and text genuinely occupy real
sequence positions, you can directly slice out "how much does image token $i$ attend to text
token $j$" as a literal sub-block of the attention matrix — which is exactly the mechanism
the attention-alignment side-experiment (not covered in depth in this notebook) is built on.

Other implementation details, each a small deliberate choice:
- **RMSNorm**, not LayerNorm (fewer parameters, no mean-subtraction, standard in recent
  LLM/DiT work).
- **2D RoPE** (rotary position embeddings) for the image tokens only — time and text tokens
  get a zero rotation angle, i.e. no positional bias, since "position in the caption" isn't
  meaningful the way "position in the image grid" is.
- **QK-norm**: an RMSNorm applied to queries and keys before the attention dot-product, a
  common stabilization trick.
- **SwiGLU**, not a plain ReLU/GELU MLP, for the feed-forward blocks.


In [ ]:
from stage2.models.lightningDiT import LightningDiT
from configs.stage2 import ConditioningArchConfig

# A small instance, just to inspect shapes and count parameters -- the real pretraining
# config uses hidden_size=1152, depth=28 (much larger); this cell is about the mechanism,
# not about training a usable model.
cond_arch = ConditioningArchConfig(num_t_tokens=4, num_c_tokens=128)
toy_dit = LightningDiT(
    input_size=16, patch_size=1, in_channels=32, hidden_size=384, depth=12, num_heads=6,
    condition_type="text", context_dim=1024, cond_arch=cond_arch,
).to(device)

n_params = sum(p.numel() for p in toy_dit.parameters())
print(f"Params: {n_params / 1e6:.1f}M   Image tokens: {toy_dit.x_embedder.num_patches}")

B = 2
x = torch.randn(B, 32, 16, 16, device=device)
t = torch.rand(B, device=device)
context = torch.randn(B, 128, 1024, device=device)
attn_mask = torch.ones(B, 128, device=device)

with torch.no_grad():
    v_pred = toy_dit(x, t, context=context, attn_mask=attn_mask)
print(f"input latent: {tuple(x.shape)}  ->  predicted velocity: {tuple(v_pred.shape)}  (same shape, as expected)")


## 7. Putting it together: the full training loss

Every training step, per micro-batch:

$$\mathcal{L} = \underbrace{\| v_\theta(x_t, t, \text{caption}) - v_t \|^2}_{\text{flow-matching loss (Sec. 2)}} \;+\; \lambda_{\text{repa}} \underbrace{\| \text{proj}(h_4) - f_{\text{DINOv2}}(x_1) \|^2}_{\text{REPA loss (Sec. 5)}}$$

where $h_4$ is the DiT's own hidden state at layer 4. This is computed in
`Transport.training_losses` and both terms are summed before a single `backward()` call —
gradients from the REPA term flow back through exactly the same network, all the way to the
embedders, alongside the main denoising gradient.


## 8. Data pipeline: get the data and precompute

Both datasets used in this notebook — the 26k-image pretraining set and the 2k-image SFT
set — are provided as ready-to-use WebDataset shards. This section gets you from those raw
shards to the precomputed latents caches (`PRETRAIN_LATENTS_DIR`, `SFT_LATENTS_DIR`) that
Sections 9 and 10 train from.

### 8.1 Download the raw WebDataset shards

```bash
# <-- point these at wherever the datasets are hosted -->
wget <DATA_URL>/dogs_recaptioned_wds.tar.gz -O /tmp/pretrain_wds.tar.gz
mkdir -p $RECAPTIONED_WDS_DIR && tar -xzf /tmp/pretrain_wds.tar.gz -C $RECAPTIONED_WDS_DIR

wget <DATA_URL>/dogs_synthetic_2k_wds.tar.gz -O /tmp/sft_wds.tar.gz
mkdir -p $SFT_WDS_DIR && tar -xzf /tmp/sft_wds.tar.gz -C $SFT_WDS_DIR
```

Each is a plain WebDataset — `.tar` shards where every sample is a `{key}.jpg` + `{key}.txt`
pair:

```bash
$ tar -tf $RECAPTIONED_WDS_DIR/shard-00000.tar | head -4
afghan_hound/n02088094_10.jpg
afghan_hound/n02088094_10.txt
afghan_hound/n02088094_1003.jpg
afghan_hound/n02088094_1003.txt
```

### 8.2 Precompute latents

**Why precompute latents offline instead of running the VAE/text-encoder/DINOv2 forward
passes every training step?** Those three models are frozen — their outputs for a given
image+caption never change during stage-2 training. Recomputing them on every single epoch
is pure wasted compute (three extra forward passes per sample, every epoch, for the entire
training run). `scripts/precompute_latents.py` runs each frozen model **once** over the
whole dataset and caches the results — VAE latent, per-token text embedding, attention
mask, and DINOv2 patch features — into new WebDataset shards. Stage-2 training then just
reads these cached tensors directly, no frozen-model forward passes needed at train time at
all.

Run it once per dataset — pretraining:

```bash
uv run torchrun --nproc_per_node=4 scripts/precompute_latents.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-EMA0.9995.yaml \
    --input-dir $RECAPTIONED_WDS_DIR \
    --output-dir $PRETRAIN_LATENTS_DIR \
    --batch-size 16
```

and SFT:

```bash
uv run torchrun --nproc_per_node=4 scripts/precompute_latents.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-SFT-dogs-synthetic-EMA0.995.yaml \
    --input-dir $SFT_WDS_DIR \
    --output-dir $SFT_LATENTS_DIR \
    --batch-size 16
```

At train time, `DogsLatentsWebDataset` (`src/data/dogs_latents_wds_dataset.py`) reads these
shards directly — `dataset.type: 'latents'` in the configs selects this path instead of
decoding raw images. Point each config's `dataset.data_dir` at the matching latents
directory (`$PRETRAIN_LATENTS_DIR` for Section 9, `$SFT_LATENTS_DIR` for Section 10) before
moving on.


## 9. Experiment 1: pretraining LightningDiT from scratch

Config:
`configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-EMA0.9995.yaml`

Key settings, and *why*:

| Setting | Value | Why |
|---|---|---|
| `training.epochs` | 200 | full pretraining budget |
| `training.global_batch_size` | 256 | split across 4 GPUs |
| `training.ema_decay` | 0.9995 | slow-moving EMA of weights, used for sampling/eval — smooths out training noise |
| `optimizer.lr` / `scheduler.base_lr` | 1e-4 | `scheduler.base_lr` is the one that actually takes effect (it overwrites the optimizer's LR right after construction) — keep both in sync in the config to avoid confusion |
| `scheduler.warmup_epochs` | 100 | long warmup, appropriate for training a large model from scratch |
| `scheduler.final_lr` | 1e-5 | linear decay target — LR ends at 1/10th of `base_lr` by the end of the schedule |
| `repa.use_repa` | true | Section 5 |

```bash
export EXPERIMENT_NAME=my-dit-pretrain-run-1     # <-- name your own run
export ENTITY=your-wandb-entity
export PROJECT=my-dogs-lightningdit

CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun --standalone --nproc_per_node=4 \
    src/train.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-EMA0.9995.yaml \
    --precision bf16 \
    --wandb
```

This writes checkpoints to `ckpts/$EXPERIMENT_NAME/checkpoints/ep-XXXXXXX.pt` — the epoch
you pick from your own run is what feeds into Section 10. On 4x RTX 3090s, my original run
took [fill in the wall-clock time you observe] for the full 200-epoch schedule; expect
roughly proportionally longer on fewer/smaller GPUs. Watch the loss and REPA-loss curves in
Weights & Biases while it runs.


## 10. Experiment 2: supervised fine-tuning (SFT) — and a lesson in what FID actually measures

**Why fine-tune at all?** The pretrained model has learned dogs and captions broadly, from a
large, noisy, wide-coverage dataset. A much smaller, more tightly curated set — 2,000
synthetic images across the same 20 breeds, 100 per breed — lets you specialize the general
model onto a narrower, cleaner distribution. `SFT_LATENTS_DIR` was already precomputed in
Section 8.2 — nothing further to prepare here, just point the config at it.

Config:
`configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-SFT-dogs-synthetic-EMA0.995.yaml`
— structurally identical to the pretrain config, with three differences worth noting:

- `dataset.data_dir` points at your small SFT latents cache (`$SFT_LATENTS_DIR`) instead of
  the large pretrain one.
- Much shorter LR schedule and smaller `ema_decay` (0.995 vs 0.9995) — appropriate for
  fine-tuning rather than training from scratch.
- Launched with `--ckpt <your pretrained checkpoint from Section 9> --init-weights-only`:
  load the pretrained model+EMA weights, but start the optimizer and epoch/step counters
  fresh, rather than resuming the original training run's schedule (which was tuned for a
  different dataset size and step count).

```bash
export EXPERIMENT_NAME=my-dit-sft-run-1     # <-- name your own run
export ENTITY=your-wandb-entity
export PROJECT=my-sft-dogs-lightningdit

CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun --standalone --nproc_per_node=4 \
    src/train.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-SFT-dogs-synthetic-EMA0.995.yaml \
    --ckpt ckpts/my-dit-pretrain-run-1/checkpoints/ep-0000100.pt \
    --init-weights-only \
    --precision bf16 \
    --wandb
```

### The lesson: watch your evaluation metric, not just the training loss

Your SFT run logs FID and Inception Score to a CSV every `eval.eval_interval` steps (see
Section 11 for exactly how) — `results/evals/.../{EXPERIMENT_NAME}_ema.csv`. Once your run
has produced a few of these evaluation points, load and plot them here:


In [ ]:
import csv
import matplotlib.pyplot as plt

# Point this at your own run's eval CSV, written automatically during Section 10's training
# (results/evals/.../{EXPERIMENT_NAME}_ema.csv) -- there is no pre-supplied data here, this
# cell only produces a plot once you have actually trained your own SFT run.
eval_csv_path = CKPT_ROOT.parent / "results" / "evals" / "my-dit-sft-run-1_ema.csv"  # <-- adjust to your path

with open(eval_csv_path) as f:
    rows = list(csv.DictReader(f))
steps = [int(r["step"]) for r in rows]
fid = [float(r["fid"]) for r in rows]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(steps, fid, marker="o")
ax.set_xlabel("training step")
ax.set_ylabel("FID (lower = better)")
ax.set_title("FID vs. training step -- your SFT run")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**What to look for.** In my own runs of this exact setup, FID rose *monotonically*
well past epoch 50 in every 100-epoch SFT run. Watch your own curve as it comes in: does it
also turn upward, and if so, around what epoch? Before jumping to "the model got worse,"
though, check one field in your own SFT config: `eval.datasets.dogs.reference_npz`. In the
config above, it's set to the *same* file used to evaluate the pretraining run in Section 9 —
`dogs_recaptioned_all_stats.npz`, built once from the 26,000-image pretraining photo set, and
never updated for what SFT is actually training toward.

That matters a lot, because FID doesn't measure "image quality" in the abstract — it measures
the Fréchet distance between your generated images' Inception-feature distribution and
*whichever reference distribution you hand it*. The SFT set (2,000 curated synthetic images)
is, by design, a different distribution from the 26,000 recaptioned real photos. If SFT is
doing exactly what it's supposed to — successfully specializing the model toward the SFT
set's own style — the generated images' feature distribution moves *toward* that set and
*away from* the original 26k-photo distribution. The fixed, mismatched reference reports that
as a rising FID, regardless of whether the images are actually getting worse. My best read of
my own results is that this reference mismatch, not overfitting, is doing most of the work
here: Section 12's reward-model comparison scores the *final* SFT checkpoint — the single
worst point on this FID curve — and both learned preference models still prefer it strongly
over the pretrained baseline. That's hard to square with "the images got worse," and easy to
square with "FID was measuring distance to the wrong reference."

None of that rules out some genuine overfitting also happening on only 2,000 training images
— it's a real risk worth taking seriously, and the two effects aren't mutually exclusive — but
it means you can't read that off the FID curve alone, because it was never given a reference
that matches what you're actually fine-tuning toward. The clean way to separate the two is in
Section 12.5's suggested exercise: build a reference from the SFT set's own images and see how
much of this curve survives.

**The practical takeaway, and a good exercise for students:** whatever the cause, this is
exactly why you evaluate on a real metric during training, not just the training loss — and
why you should know precisely what that metric is being compared against before trusting its
direction. The training loss alone would very plausibly *still be decreasing* in this regime
(the model is still fitting its training set better) while FID climbs, so loss alone would
never have surfaced this question at all.


## 11. Inference: generating images from a trained checkpoint

`scripts/generate_dog_comparison.py` loads a checkpoint's EMA weights and generates images
for a fixed list of (breed, caption) pairs — useful for consistent, repeatable visual
comparisons across checkpoints/experiments, since every checkpoint sees the exact same
prompts and the same random seed:

```bash
uv run python scripts/generate_dog_comparison.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-SFT-dogs-synthetic-EMA0.995.yaml \
    --checkpoint ckpts/my-dit-sft-run-1/checkpoints/ep-0000050.pt \
    --captions-json captions_40.json \
    --output-dir results/my_comparison \
    --seed 0
```

Internally this loads the EMA weights, builds the flow-matching ODE sampler
(`sampler.num_steps: 50`), applies classifier-free guidance (Section 2) at
`guidance.cfg.scale: 6.0`, and decodes the resulting latents back to images through the same
Stage-1 VAE from Section 3.

**Evaluation metrics.** FID (Fréchet Inception Distance) compares the distribution of
generated images against a reference set of real images in a pretrained Inception network's
feature space — lower means the generated distribution looks statistically closer to real
images. Inception Score measures both how confidently recognizable individual generated
images are *and* how diverse the generated set is overall — higher is better. Both are
computed automatically during training (`eval.eval_interval`) and are what produced the
numbers plotted in Section 10.

**For reference only** (my own run, generated exactly this way, on my own trained checkpoints
— not something you can load or reproduce exactly, but useful to
see what "in the right ballpark" looks like): a live, browsable side-by-side comparison of a
pretrained vs. SFT checkpoint, on 40 fixed captions across 20 breeds, is available here:
**https://claude.ai/code/artifact/115ad501-5b79-4131-a5fc-bf162595548e**. Once you've trained
your own two checkpoints, running the command above against each and comparing the outputs
yourself is the actual exercise.


## 12. Beyond FID/IS: scoring your images with learned human-preference models

FID and Inception Score (Sections 10-11) compare *distributions* of images against reference
statistics — they never ask "would a person actually prefer this image for this caption."
Since 2023, an active line of work trains models specifically to predict human preference
between two images for the same prompt: **PickScore** (Kirstain et al., 2023, trained on the
Pick-a-Pic dataset of real user preference picks) and **HPSv2** (Wu et al., 2023, trained on
798k binary preference choices over 434k images), with a further iteration, **HPSv3** (2025),
showing this is still an active area. Both are CLIP-backbone models fine-tuned to output a
score (or, given two images, a preference probability) — a genuinely different question from
"does this look statistically like the reference set."

This section has you score the same two sets of generated images from Section 11 (pretrained
vs. SFT checkpoint, same captions, same seed) with both models, and compare what they say
against what FID/IS already told you.

### 12.1 Generate a larger, reusable caption set

The 40 fixed captions from Section 11 work for a quick visual check, but a meaningful
preference comparison benefits from more samples. `scripts/generate_captions_RM.py`
(adapted from `scripts/generate_dog_captions.py`, Section 8's caption-invention approach)
generates a configurable number of synthetic captions per breed:

```bash
export OPENAI_API_KEY=sk-...
uv run python scripts/generate_captions_RM.py \
    --output-dir reward_model/breeds \
    --count-per-breed 25
```

(25/breed x 20 breeds = 500 total; adjust `--count-per-breed` for a smaller/larger set.) Then
collapse the per-breed `.txt` files into the single JSON `generate_dog_comparison.py` expects:

```bash
uv run python scripts/build_captions_json_from_dir.py \
    --captions-dir reward_model/breeds \
    --output-json reward_model/captions.json
```

### 12.2 Generate images from both checkpoints

Same script as Section 11, now with `--group-by-breed` so the output mirrors the caption
folder layout, and pointed at the larger caption set:

```bash
uv run python scripts/generate_dog_comparison.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-EMA0.9995.yaml \
    --checkpoint ckpts/my-dit-pretrain-run-1/checkpoints/ep-0000100.pt \
    --captions-json reward_model/captions.json \
    --output-dir reward_model/images/pretrained \
    --group-by-breed --seed 0

uv run python scripts/generate_dog_comparison.py \
    --config configs/stage2/training/t2i/t2i-ddt-en28d1152hd72-dn2d2048hd128-e2e-invae-vpred-t4-RePA-latents-SFT-dogs-synthetic-EMA0.995.yaml \
    --checkpoint ckpts/my-dit-sft-run-1/checkpoints/ep-0000050.pt \
    --captions-json reward_model/captions.json \
    --output-dir reward_model/images/sft \
    --group-by-breed --seed 0
```

### 12.3 Score with HPSv2

```bash
pip install hpsv2
```


In [ ]:
import hpsv2
from pathlib import Path

PRETRAINED_IMG_DIR = Path("reward_model/images/pretrained")
SFT_IMG_DIR = Path("reward_model/images/sft")

def hps_scores_for_dir(img_dir):
    """hpsv2.score() compares images generated by the SAME prompt -- call it once per
    (image, its own caption) pair, not once for a whole folder at once."""
    scores = {}
    for breed_dir in sorted(p for p in img_dir.iterdir() if p.is_dir()):
        breed = breed_dir.name
        scores[breed] = []
        for png_path in sorted(breed_dir.glob("*.png")):
            caption = png_path.with_suffix(".txt").read_text(encoding="utf-8").strip()
            score = hpsv2.score([str(png_path)], caption, hps_version="v2.1")[0]
            scores[breed].append(float(score))
    return scores

hps_pretrained = hps_scores_for_dir(PRETRAINED_IMG_DIR)
hps_sft = hps_scores_for_dir(SFT_IMG_DIR)

all_pre = [s for v in hps_pretrained.values() for s in v]
all_sft = [s for v in hps_sft.values() for s in v]
print(f"HPSv2 mean -- pretrained: {sum(all_pre)/len(all_pre):.4f}   sft: {sum(all_sft)/len(all_sft):.4f}")


### 12.4 Score with PickScore

PickScore doesn't need its own repo cloned -- it's a fine-tuned CLIP checkpoint, usable
directly through `transformers`. Given two images for the *same* prompt, it returns a
preference probability for each (softmax over the pair), rather than an independent score
per image the way HPSv2 does above.


In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch

processor = AutoProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
pickscore_model = AutoModel.from_pretrained("yuvalkirstain/PickScore_v1").eval().to(device)

@torch.no_grad()
def pickscore_probs(prompt, images):
    image_inputs = processor(images=images, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
    text_inputs = processor(text=prompt, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
    image_embs = pickscore_model.get_image_features(**image_inputs)
    image_embs = image_embs / torch.norm(image_embs, dim=-1, keepdim=True)
    text_embs = pickscore_model.get_text_features(**text_inputs)
    text_embs = text_embs / torch.norm(text_embs, dim=-1, keepdim=True)
    scores = pickscore_model.logit_scale.exp() * (text_embs @ image_embs.T)[0]
    return torch.softmax(scores, dim=-1).cpu().tolist()

sft_wins, total = 0, 0
for breed_dir in sorted(p for p in PRETRAINED_IMG_DIR.iterdir() if p.is_dir()):
    breed = breed_dir.name
    for pre_png in sorted(breed_dir.glob("*.png")):
        sft_png = SFT_IMG_DIR / breed / pre_png.name
        caption = pre_png.with_suffix(".txt").read_text(encoding="utf-8").strip()
        prob_pre, prob_sft = pickscore_probs(caption, [Image.open(pre_png), Image.open(sft_png)])
        total += 1
        sft_wins += prob_sft > 0.5

print(f"PickScore: SFT preferred in {sft_wins}/{total} pairs ({sft_wins/total*100:.1f}%)")


### 12.5 What to look for

I ran this exact pipeline (500 captions, pretrained vs. an SFT checkpoint) and
found something worth thinking through rather than taking at face value: **both HPSv2 and
PickScore consistently and strongly preferred the SFT checkpoint over the pretrained
baseline** — PickScore favored SFT in the large majority of pairs, consistently across
nearly every breed, and HPSv2's mean score was clearly higher for SFT too.

At first glance this can look like it *contradicts* Section 10's finding (FID rising past
epoch 50 as SFT continues) — but it's strong supporting evidence for the explanation Section
10 already landed on, not a contradiction. The SFT checkpoint scored above (`ep-0000100`) is
the *final* checkpoint of its run — the single worst point on Section 10's FID curve — yet
it's still the one both reward models prefer strongly over pretrained. That combination is
hard to explain as "the images got worse the longer SFT ran," and easy to explain as "FID was
measuring distance to the wrong reference distribution the whole time." Preference models
have no such reference-distribution dependency: they score each image (and pair) directly, so
they're not vulnerable to the same artifact.

This is also *why* the two metrics are answering genuinely different questions, not just
disagreeing: FID asks "does this distribution statistically match a fixed reference set,"
while a preference-model comparison asks "would a human generally prefer this image over that
one." A model can legitimately move away from one specific reference distribution while
getting more preferred overall — those aren't in tension once you see FID as reference-
relative rather than an absolute quality score. A cleaner (harder) experiment worth trying
yourself, to check this isn't just a plausible story: `scripts/compute_dogs_val_stats.py`
builds exactly this kind of reference `.npz` (InceptionV3 mean/covariance) from any
WebDataset folder of real images — point it at your own SFT image set, swap the new file into
`reference_npz` in the SFT config, and re-run Section 11's FID eval against *that* reference
instead. If the mismatch explanation is right, that curve should look very different from
Section 10's.


## 13. Key takeaways

1. **Latent-space diffusion** trades a small, fixed amount of reconstruction fidelity (the
   frozen VAE) for a drastic reduction in sequence length, making transformer-based image
   generation computationally tractable.
2. **Flow matching** trains a network to predict the velocity along a straight-line path
   between noise and data — a simple MSE regression objective, with sampling done by
   numerically integrating the learned velocity field.
3. **Conditioning doesn't have to mean AdaLN.** In-context (token-based) conditioning is a
   real, working alternative used in this codebase — time and text become literal sequence
   positions, letting a single shared self-attention mechanism handle all conditioning.
4. **REPA-style auxiliary losses** can pull a generative model's internal features toward a
   strong, independently-trained self-supervised signal (DINOv2), separately from the main
   generative objective.
5. **Precomputing frozen-model outputs offline** (Section 8) is a straightforward, large
   compute saving whenever a training pipeline includes frozen components.
6. **A metric moving in the "wrong" direction doesn't automatically mean the model got
   worse — know what it's being compared against.** Section 10's real FID curve rises
   throughout SFT, but the evidence points to this being mostly a reference-distribution
   artifact (the same pretraining-photo reference gets reused to evaluate a model being
   fine-tuned toward a deliberately different, synthetic distribution), not a straightforward
   quality regression. Training loss alone would never have surfaced this question at all —
   but neither would blindly trusting FID's *direction* without checking what it's measuring
   distance to.
7. **One metric is never the complete picture.** Section 12's real result — FID says SFT
   drifts away from the pretraining reference the longer it trains, while PickScore/HPSv2 say
   the *same* final checkpoint is clearly preferred over never fine-tuning at all — isn't a
   contradiction, it's two structurally different questions: distance to one fixed reference
   set vs. learned human preference. Reconciling them, rather than picking whichever one
   agrees with your prior, is the actual skill.

## Suggested exercises

- Reproduce the FID-vs-epoch plot in Section 10 for a checkpoint you train yourself.
- Implement a simple FID-based early-stopping rule (e.g. stop when eval FID hasn't improved
  in $N$ consecutive evaluations), apply it to a new SFT run, then score *that* early-stopped
  checkpoint with Section 12's reward models against the final, full-length checkpoint — given
  Section 10's finding, does stopping early on FID actually give you the more human-preferred
  checkpoint, or does it stop too soon precisely because of the reference-mismatch effect?
- Try `repa_layer_depth` values other than 4 (Section 5) — does aligning at an earlier or
  later layer change convergence speed?
- Train once with `repa.use_repa: false` and once `true`, holding everything else fixed —
  reproduce (or fail to reproduce) REPA's claimed benefit on this dataset.
- Read `LightningDiT._build_sequence` (Section 6) and draw the full token sequence layout
  by hand (which positions are image tokens, which are time, which are text) before running
  any code — then verify your diagram against `x_embedder.num_patches`,
  `cond_arch.num_t_tokens`, and `cond_arch.num_c_tokens`.
- Find a caption/breed where PickScore and HPSv2 *disagree* on which checkpoint is better
  (Section 12) and look at the two images yourself — what does each metric seem to be
  picking up on that the other isn't?
- Plot PickScore win-rate per breed (Section 12.3 already computes the per-pair result) —
  is the SFT preference uniform across breeds, or concentrated in a few?
- Build a reference `.npz` from your SFT set's own images with
  `scripts/compute_dogs_val_stats.py`, and re-run Section 11's FID eval against it instead of
  the pretraining reference — how much of Section 10's "overfitting" FID rise survives once
  the reference distribution actually matches what SFT was trained on?

## References

- Lipman et al., *Flow Matching for Generative Modeling*, 2023.
- Peebles & Xie, *Scalable Diffusion Models with Transformers (DiT)*, 2022.
- Ho & Salimans, *Classifier-Free Diffusion Guidance*, 2022.
- Yu et al., *Representation Alignment for Generation (REPA)*, 2024.
- Oquab et al., *DINOv2: Learning Robust Visual Features without Supervision*, 2023.
- Kirstain et al., *Pick-a-Pic: An Open Dataset of User Preferences for Text-to-Image Generation (PickScore)*, 2023.
- Wu et al., *Human Preference Score v2: A Solid Benchmark for Evaluating Human Preferences of Text-to-Image Synthesis*, 2023.
- Qwen Team, *Qwen3 Technical Report*, 2026.
